# Paradigmata strojového učení: 8 jednoduchých příkladů

Tento dokument poskytuje jednoduché příklady pro 8 klíčových paradigmat strojového učení, s vlastními datovými sadami a kompletními implementacemi v Pythonu.

## 1. Regrese

### Popis
Regrese předpovídá spojité numerické hodnoty na základě vstupních příznaků. Je užitečná pro předpovědi, analýzu trendů a porozumění vztahům mezi proměnnými.

**Aplikace:** Predikce cen, prognóza prodeje, odhad fyzikálních vztahů, predikce růstu

### Příklad: Predikce cen nemovitostí

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Vytvoření syntetického datasetu
np.random.seed(42)
n_samples = 200

# Příznaky: velikost domu (čtvereční stopy), ložnice, stáří (roky), vzdálenost od centra města (míle)
X = np.random.rand(n_samples, 4)  
X[:, 0] = X[:, 0] * 3000 + 1000  # Velikost domu: 1000-4000 čtverečních stop
X[:, 1] = np.round(X[:, 1] * 4 + 1)  # Ložnice: 1-5
X[:, 2] = np.round(X[:, 2] * 50)  # Stáří: 0-50 let
X[:, 3] = X[:, 3] * 30  # Vzdálenost: 0-30 mil

# Cílová proměnná: cena domu (tisíce $)
y = 100 + 0.2 * X[:, 0] - 40 * X[:, 3] + 15 * X[:, 1] - X[:, 2] * 2
y += np.random.randn(n_samples) * 50  # Přidání šumu

# Vytvoření DataFrame
df = pd.DataFrame(X, columns=['Velikost', 'Loznice', 'Stari', 'Vzdalenost'])
df['Cena'] = y

# Zobrazení vzorku dat
print(df.head())
print("\nStatistika datasetu:")
print(df.describe())

# Vizualizace vztahů
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.scatter(df['Velikost'], df['Cena'], alpha=0.6)
plt.title('Velikost domu vs Cena')
plt.xlabel('Velikost (čtverečních stop)')
plt.ylabel('Cena (tisíc $)')

plt.subplot(2, 2, 2)
plt.scatter(df['Vzdalenost'], df['Cena'], alpha=0.6)
plt.title('Vzdálenost od centra města vs Cena')
plt.xlabel('Vzdálenost (míle)')
plt.ylabel('Cena (tisíc $)')

plt.subplot(2, 2, 3)
plt.scatter(df['Stari'], df['Cena'], alpha=0.6)
plt.title('Stáří domu vs Cena')
plt.xlabel('Stáří (roky)')
plt.ylabel('Cena (tisíc $)')

plt.subplot(2, 2, 4)
plt.scatter(df['Loznice'], df['Cena'], alpha=0.6)
plt.title('Počet ložnic vs Cena')
plt.xlabel('Ložnice')
plt.ylabel('Cena (tisíc $)')

plt.tight_layout()
plt.show()

# Rozdělení dat
X = df[['Velikost', 'Loznice', 'Stari', 'Vzdalenost']]
y = df['Cena']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Trénování modelu
model = LinearRegression()
model.fit(X_train, y_train)

# Predikce
y_pred = model.predict(X_test)

# Vyhodnocení modelu
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\nKoeficienty modelu: {model.coef_}")
print(f"Střední kvadratická chyba: {mse:.2f}")
print(f"Skóre R²: {r2:.2f}")

# Vizualizace predikce vs skutečnost
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Skutečná cena')
plt.ylabel('Predikovaná cena')
plt.title('Predikované vs skutečné ceny domů')
plt.show()

## 2. Klasifikace

### Popis
Klasifikace přiřazuje vstupy k předdefinovaným kategoriím nebo třídám. Pomáhá při rozhodování, kategorizaci a rozpoznávání vzorů v označených datech.

**Aplikace:** Detekce spamu, lékařská diagnostika, segmentace zákazníků, analýza sentimentu

### Příklad: Predikce odchodu zákazníků

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Vytvoření syntetického datasetu
np.random.seed(42)
n_samples = 200

# Příznaky: věk účtu (měsíce), měsíční poplatky, celkové poplatky, 
# servisní volání, tickety podpory, frekvence používání
X = np.zeros((n_samples, 6))
X[:, 0] = np.random.randint(1, 60, size=n_samples)  # Věk účtu: 1-60 měsíců
X[:, 1] = np.random.uniform(30, 120, size=n_samples)  # Měsíční poplatky: $30-$120
X[:, 2] = X[:, 0] * X[:, 1] * (0.8 + 0.4 * np.random.random(n_samples))  # Celkové poplatky
X[:, 3] = np.random.poisson(2, size=n_samples)  # Servisní volání: Poissonovo rozdělení
X[:, 4] = np.random.poisson(1, size=n_samples)  # Tickety podpory: Poissonovo rozdělení
X[:, 5] = np.random.uniform(1, 10, size=n_samples)  # Frekvence používání: škála 1-10

# Cílová proměnná: odchod (0: neodešel, 1: odešel)
# Vyšší pravděpodobnost odchodu při více servisních voláních, více ticketech, vyšších poplatcích, nižším používání
logits = -2 + 0.4 * X[:, 3] + 0.5 * X[:, 4] + 0.02 * X[:, 1] - 0.3 * X[:, 5] - 0.01 * X[:, 0]
probs = 1 / (1 + np.exp(-logits))
y = np.random.binomial(1, probs)

# Vytvoření DataFrame
df = pd.DataFrame(X, columns=['VekUctu', 'MesicniPoplatky', 'CelkovePoplatky', 
                             'ServisniVolani', 'TicketyPodpory', 'FrekvencePouzivani'])
df['Odchod'] = y

# Zobrazení dat
print(df.head())
print("\nDistribuce cílové proměnné:")
print(df['Odchod'].value_counts())

# Vizualizace příznaků podle třídy
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
sns.boxplot(x='Odchod', y='VekUctu', data=df)
plt.title('Věk účtu podle stavu odchodu')

plt.subplot(2, 3, 2)
sns.boxplot(x='Odchod', y='MesicniPoplatky', data=df)
plt.title('Měsíční poplatky podle stavu odchodu')

plt.subplot(2, 3, 3)
sns.boxplot(x='Odchod', y='ServisniVolani', data=df)
plt.title('Servisní volání podle stavu odchodu')

plt.subplot(2, 3, 4)
sns.boxplot(x='Odchod', y='TicketyPodpory', data=df)
plt.title('Tickety podpory podle stavu odchodu')

plt.subplot(2, 3, 5)
sns.boxplot(x='Odchod', y='FrekvencePouzivani', data=df)
plt.title('Frekvence používání podle stavu odchodu')

plt.tight_layout()
plt.show()

# Příprava dat pro modelování
X = df.drop('Odchod', axis=1)
y = df['Odchod']

# Rozdělení dat na trénovací a testovací sadu
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Škálování příznaků
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Trénování modelu
model = RandomForestClassifier(random_state=42)
model.fit(X_train_scaled, y_train)

# Predikce
y_pred = model.predict(X_test_scaled)

# Vyhodnocení modelu
print("\nKlasifikační zpráva:")
print(classification_report(y_test, y_pred))

print("\nMatice záměn:")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predikované')
plt.ylabel('Skutečné')
plt.title('Matice záměn')
plt.show()

# Důležitost příznaků
feature_importance = pd.DataFrame({
    'Příznak': X.columns,
    'Důležitost': model.feature_importances_
}).sort_values('Důležitost', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Důležitost', y='Příznak', data=feature_importance)
plt.title('Důležitost příznaků pro predikci odchodu')
plt.tight_layout()
plt.show()

## 3. Shlukování

### Popis
Shlukování seskupuje podobná datová body na základě jejich vlastností. Je užitečné pro objevování skrytých vzorů a struktur v neoznačených datech.

**Aplikace:** Segmentace zákazníků, organizace dokumentů, detekce anomálií, komprese obrazu

### Příklad: Segmentace zákazníků

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Vytvoření syntetického datasetu
np.random.seed(42)
n_samples = 200

# Vytvoření dat zákazníků s přirozenými shluky
# 3 hlavní segmenty: úsporní nákupčí, střední třída, prémiový segment
group1 = np.random.multivariate_normal(
    mean=[20, 15, 2, 1],  # Nízké útraty, nízká frekvence návštěv
    cov=[[10, 0, 0, 0], [0, 5, 0, 0], [0, 0, 1, 0], [0, 0, 0, 0.5]], 
    size=70)

group2 = np.random.multivariate_normal(
    mean=[50, 30, 5, 3],  # Střední útraty, střední frekvence návštěv
    cov=[[15, 0, 0, 0], [0, 8, 0, 0], [0, 0, 1.5, 0], [0, 0, 0, 1]], 
    size=80)

group3 = np.random.multivariate_normal(
    mean=[100, 60, 7, 5],  # Vysoké útraty, vysoká frekvence návštěv
    cov=[[20, 0, 0, 0], [0, 10, 0, 0], [0, 0, 2, 0], [0, 0, 0, 1.5]], 
    size=50)

# Kombinace skupin
X = np.vstack([group1, group2, group3])

# Vytvoření DataFrame
df = pd.DataFrame(X, columns=['Prumerna_Mesicni_Utrata', 'Hodnota_Kosiku', 
                             'Navstevy_Obchodu_Za_Mesic', 'Online_Nakupy_Za_Mesic'])

# Zajištění, že všechny hodnoty jsou kladné
df['Prumerna_Mesicni_Utrata'] = np.abs(df['Prumerna_Mesicni_Utrata'])
df['Hodnota_Kosiku'] = np.abs(df['Hodnota_Kosiku'])
df['Navstevy_Obchodu_Za_Mesic'] = np.abs(df['Navstevy_Obchodu_Za_Mesic'])
df['Online_Nakupy_Za_Mesic'] = np.abs(df['Online_Nakupy_Za_Mesic'])

# Zobrazení dat
print(df.head())
print("\nStatistika datasetu:")
print(df.describe())

# Vizualizace distribuce dat
plt.figure(figsize=(12, 10))

plt.subplot(2, 2, 1)
sns.histplot(df['Prumerna_Mesicni_Utrata'], bins=20, kde=True)
plt.title('Distribuce průměrné měsíční útraty')

plt.subplot(2, 2, 2)
sns.histplot(df['Hodnota_Kosiku'], bins=20, kde=True)
plt.title('Distribuce průměrné hodnoty košíku')

plt.subplot(2, 2, 3)
plt.scatter(df['Prumerna_Mesicni_Utrata'], df['Hodnota_Kosiku'], alpha=0.6)
plt.title('Měsíční útrata vs Hodnota košíku')
plt.xlabel('Průměrná měsíční útrata')
plt.ylabel('Průměrná hodnota košíku')

plt.subplot(2, 2, 4)
plt.scatter(df['Navstevy_Obchodu_Za_Mesic'], df['Online_Nakupy_Za_Mesic'], alpha=0.6)
plt.title('Návštěvy obchodu vs Online nákupy')
plt.xlabel('Návštěvy obchodu za měsíc')
plt.ylabel('Online nákupy za měsíc')

plt.tight_layout()
plt.show()

# Škálování příznaků
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# Určení optimálního počtu shluků
inertia = []
silhouette_scores = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Vykreslení metody lokte a silhouette skóre
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(k_range, inertia, 'o-')
plt.xlabel('Počet shluků')
plt.ylabel('Inertia')
plt.title('Metoda lokte')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(k_range, silhouette_scores, 'o-')
plt.xlabel('Počet shluků')
plt.ylabel('Silhouette skóre')
plt.title('Metoda silhouette')
plt.grid(True)

plt.tight_layout()
plt.show()

# Aplikace KMeans s optimálním k (3 pro tento příklad)
k = 3
kmeans = KMeans(n_clusters=k, random_state=42)
df['Shluk'] = kmeans.fit_predict(X_scaled)

# Vizualizace shluků
plt.figure(figsize=(14, 7))

plt.subplot(1, 2, 1)
sns.scatterplot(x='Prumerna_Mesicni_Utrata', y='Hodnota_Kosiku', hue='Shluk', 
                palette='viridis', data=df, s=70, alpha=0.7)
plt.title('Segmenty zákazníků: Útrata vs Hodnota košíku')

plt.subplot(1, 2, 2)
sns.scatterplot(x='Navstevy_Obchodu_Za_Mesic', y='Online_Nakupy_Za_Mesic', 
                hue='Shluk', palette='viridis', data=df, s=70, alpha=0.7)
plt.title('Segmenty zákazníků: Návštěvy vs Online nákupy')

plt.tight_layout()
plt.show()

# Analýza charakteristik shluků
cluster_analysis = df.groupby('Shluk').mean()
print("\nAnalýza shluků:")
print(cluster_analysis)

# Vizualizace profilů shluků
plt.figure(figsize=(14, 6))
cluster_analysis_normalized = cluster_analysis / cluster_analysis.max()
sns.heatmap(cluster_analysis_normalized, annot=cluster_analysis.round(1), 
            cmap='YlGnBu', fmt='.1f', linewidths=0.5)
plt.title('Profily segmentů zákazníků')
plt.show()

## 4. Redukce dimenzionality

### Popis
Redukce dimenzionality transformuje vysokodimenzionální data do reprezentace s nižší dimenzí při zachování důležitých informací. Pomáhá s vizualizací, výpočetní efektivitou a snižováním šumu.

**Aplikace:** Vizualizace dat, extrakce příznaků, zpracování obrazu, redukce šumu

### Příklad: Vizualizace vlastností produktů

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.decomposition import PCA
# from sklearn.manifold import TSNE
# from sklearn.preprocessing import StandardScaler

# # Vytvoření syntetického datasetu - vlastnosti produktů
# np.random.seed(42)
# n_samples = 200

# # Generování dat pro 10 vlastností produktů (dimenzí)
# X = np.zeros((n_samples, 10))

# # Vytvoření 3 základních faktorů, které ovlivní 10 pozorovaných vlastností
# # Faktor 1: Kvalita/Prémiové vlastnosti
# factor1 = np.random.normal(0, 1, n_samples)
# # Faktor 2: Použitelnost/Pohodlí
# factor2 = np.random.normal(0, 1, n_samples)
# # Faktor 3: Inovace/Unikátnost
# factor3 = np.random.normal(0, 1, n_samples)

# # Generování 10 vlastností na základě 3 základních faktorů
# # Vlastnosti 0-3 jsou ovlivněny faktorem 1 (Kvalita)
# X[:, 0] = 5 + 2 * factor1 + 0.5 * np.random.normal(0, 1, n_samples)  # Odolnost
# X[:, 1] = 7 + 1.8 * factor1 + 0.3 * factor2 + 0.5 * np.random.normal(0, 1, n_samples)  # Kvalita materiálu
# X[:, 2] = 6 + 1.5 * factor1 + 0.8 * np.random.normal(0, 1, n_samples)  # Kvalita povrchu
# X[:, 3] = 8 + 1.2 * factor1 + 0.4 * factor3 + 0.6 * np.random.normal(0, 1, n_samples)  # Záruka

# # Vlastnosti 4-6 jsou ovlivněny faktorem 2 (Použitelnost)
# X[:, 4] = 6 + 1.7 * factor2 + 0.4 * factor1 + 0.7 * np.random.normal(0, 1, n_samples)  # Snadnost použití
# X[:, 5] = 7 + 1.5 * factor2 + 0.5 * np.random.normal(0, 1, n_samples)  # Uživatelské rozhraní
# X[:, 6] = 4 + 1.3 * factor2 + 0.2 * factor1 + 0.6 * np.random.normal(0, 1, n_samples)  # Křivka učení

# # Vlastnosti 7-9 jsou ovlivněny faktorem 3 (Inovace)
# X[:, 7] = 5 + 1.8 * factor3 + 0.3 * factor1 + 0.7 * np.random.normal(0, 1, n_samples)  # Unikátní funkce
# X[:, 8] = 6 + 1.6 * factor3 + 0.2 * factor2 + 0.6 * np.random.normal(0, 1, n_samples)  # Inovace designu
# X[:, 9] = 4 + 1.4 * factor3 + 0.5 * np.random.normal(0, 1, n_samples)  # Technologická úroveň

# # Vytvoření kategorií produktů (pro barevné odlišení ve vizualizaci)
# # 0: Domácnost, 1: Elektronika, 2: Nářadí, 3: Nábytek
# categories = np.zeros(n_samples, dtype=int)
# categories[50:100] = 1
# categories[100:150] = 2
# categories[150:] = 3

# # Vytvoření DataFrame
# feature_names = ['Odolnost', 'Kvalita_Materialu', 'Kvalita_Povrchu', 'Zaruka',
#                 'Snadnost_Pouziti', 'Uzivatelske_Rozhrani', 'Krivka_Uceni',
#                 'Unikatni_Funkce', 'Inovace_Designu', 'Technologicka_Uroven']

# category_names = ['Domácnost', 'Elektronika', 'Nářadí', 'Nábytek']
# df = pd.DataFrame(X, columns=feature_names)
# df['Kategorie'] = [category_names[c] for c in categories]

# # Zobrazení dat
# print(df.head())
# print("\nStatistika datasetu:")
# print(df.describe())

# # Vizualizace korelace vlastností
# plt.figure(figsize=(12, 10))
# sns.heatmap(df.iloc[:, :-1].corr(), annot=True, cmap='coolwarm', fmt='.2f')
# plt.title('Korelační matice vlastností')
# plt.tight_layout()
# plt.show()

# # Škálování vlastností pro redukci dimenzionality
# X = df.iloc[:, :-1].values
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# # Aplikace PCA
# pca = PCA()
# X_pca = pca.fit_transform(X_scaled)

# # Vykreslení vysvětleného rozptylu
# explained_variance = pca.explained_variance_ratio_
# cumulative_variance = np.cumsum(explained_variance)

# plt.figure(figsize=(10, 6))
# plt.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7, 
#         label='Individuální vysvětlený rozptyl')
# plt.plot(range(1, len(explained_variance) + 1), cumulative_variance, 'r-o', 
#          label='Kumulativní vysvětlený rozptyl')
# plt.axhline(y=0.8, color='k', linestyle='--', label='Práh 80 %')
# plt.xlabel('Hlavní komponenty')
# plt.ylabel('Poměr vysvětleného rozptylu')
# plt.title('PCA - Vysvětlený rozptyl')
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()

# # Vytvoření 2D vizualizace pomocí PCA
# plt.figure(figsize=(12, 5))

# plt.subplot(1, 2, 1)
# plt.scatter(X_pca[:, 0], X_pca[:, 1], c=[['blue', 'green', 'red', 'purple'][cat] for cat in categories], alpha=0.7)
# plt.xlabel('První hlavní komponenta')
# plt.ylabel('Druhá hlavní komponenta')
# plt.title('PCA: První dvě komponenty')

# # Váhy vlastností na první dvě komponenty
# loadings = pca.components_.T[:, :2]
# for i, feature in enumerate(feature_names):
#     plt.arrow(0, 0, loadings[i, 0]*5, loadings[i, 1]*5, color='k', alpha=0.5)
#     plt.text(loadings[i, 0]*5.2, loadings[i, 1]*5.2, feature, color='k', ha='center', va='center')
    
# plt.grid(True)

# # Aplikace t-SNE pro porovnání
# tsne = TSNE(n_components=2, random_state=42)
# X_tsne = tsne.fit_transform(X_scaled)

# plt.subplot(1, 2, 2)
# scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], 
#                       c=[['blue', 'green', 'red', 'purple'][cat] for cat in categories], alpha=0.7)
# plt.xlabel('t-SNE vlastnost 1')
# plt.ylabel('t-SNE vlastnost 2')
# plt.title('t-SNE: 2D zobrazení')

# categories_list = df['Kategorie'].unique()
# plt.legend(handles=scatter.legend_elements()[0], labels=categories_list)
# plt.grid(True)

# plt.tight_layout()
# plt.show()

# # První 3 komponenty pro 3D vizualizaci
# fig = plt.figure(figsize=(10, 8))
# ax = fig.add_subplot(111, projection='3d')
# scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], 
#                      c=[['blue', 'green', 'red', 'purple'][cat] for cat in categories], alpha=0.7)
# ax.set_xlabel('PC1')
# ax.set_ylabel('PC2')
# ax.set_zlabel('PC3')
# ax.set_title('PCA: První tři hlavní komponenty')
# plt.legend(handles=scatter.legend_elements()[0], labels=categories_list)
# plt.tight_layout()
# plt.show()

# # Analýza vah komponent
# top_features = pd.DataFrame(
#     pca.components_[:3].T,
#     columns=['PC1', 'PC2', 'PC3'],
#     index=feature_names
# )
# print("\nVáhy PCA komponent:")
# print(top_features)

# plt.figure(figsize=(12, 4))
# sns.heatmap(top_features, annot=True, cmap='coolwarm')
# plt.title('Váhy PCA komponent')
# plt.tight_layout()
# plt.show()

```
UserWarning: Collection without array used. Make sure to specify the values to be colormapped via the `c` argument.

UserWarning: Mismatched number of handles and labels: len(handles) = 0 len(labels) = 4

UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D

# Vytvoření syntetického datasetu - atributy produktů
np.random.seed(42)
n_samples = 200

# Generování dat pro 10 produktových příznaků (dimenzí)
X = np.zeros((n_samples, 10))

# Vytvoření 3 základních faktorů, které budou ovlivňovat 10 pozorovaných příznaků
faktor1 = np.random.normal(0, 1, n_samples)  # Kvalita
faktor2 = np.random.normal(0, 1, n_samples)  # Použitelnost
faktor3 = np.random.normal(0, 1, n_samples)  # Inovace

# Generování 10 příznaků na základě 3 základních faktorů
X[:, 0] = 5 + 2 * faktor1 + 0.5 * np.random.normal(0, 1, n_samples)  # Trvanlivost
X[:, 1] = 7 + 1.8 * faktor1 + 0.3 * faktor2 + 0.5 * np.random.normal(0, 1, n_samples)  # Kvalita materiálu
X[:, 2] = 6 + 1.5 * faktor1 + 0.8 * np.random.normal(0, 1, n_samples)  # Kvalita zpracování
X[:, 3] = 8 + 1.2 * faktor1 + 0.4 * faktor3 + 0.6 * np.random.normal(0, 1, n_samples)  # Záruční doba
X[:, 4] = 6 + 1.7 * faktor2 + 0.4 * faktor1 + 0.7 * np.random.normal(0, 1, n_samples)  # Jednoduchost použití
X[:, 5] = 7 + 1.5 * faktor2 + 0.5 * np.random.normal(0, 1, n_samples)  # Uživatelské rozhraní
X[:, 6] = 4 + 1.3 * faktor2 + 0.2 * faktor1 + 0.6 * np.random.normal(0, 1, n_samples)  # Křivka učení
X[:, 7] = 5 + 1.8 * faktor3 + 0.3 * faktor1 + 0.7 * np.random.normal(0, 1, n_samples)  # Jedinečné funkce
X[:, 8] = 6 + 1.6 * faktor3 + 0.2 * faktor2 + 0.6 * np.random.normal(0, 1, n_samples)  # Inovace designu
X[:, 9] = 4 + 1.4 * faktor3 + 0.5 * np.random.normal(0, 1, n_samples)  # Úroveň technologie

# Vytvoření kategorií produktů (pro obarvení vizualizací)
kategorie = np.zeros(n_samples, dtype=int)
kategorie[50:100] = 1
kategorie[100:150] = 2
kategorie[150:] = 3

# Vytvoření DataFrame
nazvy_priznaku = ['Trvanlivost', 'Kvalita_materiálu', 'Kvalita_zpracování', 'Záruční_doba',
                 'Jednoduchost_použití', 'Uživatelské_rozhraní', 'Křivka_učení',
                 'Jedinečné_funkce', 'Inovace_designu', 'Úroveň_technologie']

nazvy_kategorii = ['Domácnost', 'Elektronika', 'Nástroje', 'Nábytek']
df = pd.DataFrame(X, columns=nazvy_priznaku)
df['Kategorie'] = [nazvy_kategorii[c] for c in kategorie]

# Zobrazení dat
print(df.head())
print("\nStatistiky datasetu:")
print(df.describe())

# Vizualizace korelace příznaků
plt.figure(figsize=(12, 10))
sns.heatmap(df.iloc[:, :-1].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Korelační matice příznaků')
plt.tight_layout()
plt.show()

# Škálování příznaků
X = df.iloc[:, :-1].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Graf vysvětlené variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(10, 6))
plt.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7, label='Individuální vysvětlená variance')
plt.plot(range(1, len(explained_variance) + 1), cumulative_variance, 'r-o', label='Kumulativní vysvětlená variance')
plt.axhline(y=0.8, color='k', linestyle='--', label='80% práh')
plt.xlabel('Hlavní komponenty')
plt.ylabel('Podíl vysvětlené variance')
plt.title('PCA - Vysvětlená variance')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Barvy pro vykreslení
colors = ['blue', 'green', 'red', 'purple']
c_values = [colors[cat] for cat in kategorie]

# 2D PCA
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=c_values, alpha=0.7)
plt.xlabel('První hlavní komponenta')
plt.ylabel('Druhá hlavní komponenta')
plt.title('PCA: První dvě komponenty')

# Zatížení příznaků
loadings = pca.components_.T[:, :2]
for i, feature in enumerate(nazvy_priznaku):
    plt.arrow(0, 0, loadings[i, 0]*5, loadings[i, 1]*5, color='k', alpha=0.5)
    plt.text(loadings[i, 0]*5.2, loadings[i, 1]*5.2, feature, color='k', ha='center', va='center')
plt.grid(True)

# t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

plt.subplot(1, 2, 2)
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=c_values, alpha=0.7)
plt.xlabel('t-SNE příznak 1')
plt.ylabel('t-SNE příznak 2')
plt.title('t-SNE: 2D projekce')

# Vlastní legenda
legend_elements = [Line2D([0], [0], marker='o', color='w', label=label,
                          markerfacecolor=color, markersize=10)
                   for label, color in zip(nazvy_kategorii, colors)]
plt.legend(handles=legend_elements)
plt.grid(True)
plt.tight_layout()
plt.show()

# 3D PCA
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=c_values, alpha=0.7)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('PCA: První tři hlavní komponenty')
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

# Zatížení komponent PCA
top_features = pd.DataFrame(
    pca.components_[:3].T,
    columns=['PC1', 'PC2', 'PC3'],
    index=nazvy_priznaku
)
print("\nZatížení komponent PCA:")
print(top_features)

plt.figure(figsize=(12, 4))
sns.heatmap(top_features, annot=True, cmap='coolwarm')
plt.title('Zatížení komponent PCA')
plt.tight_layout()
plt.show()

## 5. Ensemble Learning

### Popis
Ensemble learning kombinuje více modelů ke zlepšení výkonu, stability a generalizovatelnosti. Pomáhá snižovat variabilitu, zkreslení a přeučení jednotlivých modelů.

**Aplikace:** Pokročilé predikční úlohy, zvyšování přesnosti modelů, robustní systémy strojového učení, soutěže

### Příklad: Predikce selhání úvěru

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import classification_report, roc_curve, auc, roc_auc_score
# from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# # Vytvoření syntetického datasetu úvěrového selhání
# np.random.seed(42)
# n_samples = 200

# # Generování příznaků
# X = np.zeros((n_samples, 7))
# X[:, 0] = np.random.normal(40, 12, n_samples)  # Věk: průměr 40, std 12
# X[:, 1] = np.random.normal(60000, 25000, n_samples)  # Roční příjem
# X[:, 2] = np.random.normal(15000, 7000, n_samples)  # Částka úvěru
# X[:, 3] = np.random.uniform(1, 7, n_samples)  # Doba trvání úvěru (roky)
# X[:, 4] = np.random.normal(650, 100, n_samples)  # Kreditní skóre
# X[:, 5] = np.random.poisson(3, n_samples)  # Délka zaměstnání (roky)
# X[:, 6] = np.random.binomial(1, 0.7, n_samples)  # Má předchozí úvěry (0 nebo 1)

# # Zajištění rozumných rozsahů pro příznaky
# X[:, 0] = np.clip(X[:, 0], 18, 70)  # Věk mezi 18 a 70
# X[:, 1] = np.clip(X[:, 1], 20000, 150000)  # Příjem mezi 20k a 150k
# X[:, 2] = np.clip(X[:, 2], 5000, 50000)  # Částka úvěru mezi 5k a 50k
# X[:, 4] = np.clip(X[:, 4], 400, 850)  # Kreditní skóre mezi 400 a 850
# X[:, 5] = np.clip(X[:, 5], 0, 20)  # Délka zaměstnání mezi 0 a 20 lety

# # Výpočet pravděpodobnosti selhání
# # Vyšší riziko selhání u: nižší příjem, vyšší částka úvěru, delší doba trvání, nižší kreditní skóre, kratší zaměstnání
# logits = (-3 + 
#           -0.00002 * X[:, 1] +  # Příjem (negativní efekt)
#           0.00015 * X[:, 2] +   # Částka úvěru (pozitivní efekt)
#           0.3 * X[:, 3] +       # Doba trvání úvěru (pozitivní efekt)
#           -0.01 * X[:, 4] +     # Kreditní skóre (negativní efekt)
#           -0.15 * X[:, 5])      # Délka zaměstnání (negativní efekt)

# probabilities = 1 / (1 + np.exp(-logits))
# y = np.random.binomial(1, probabilities)

# # Vytvoření DataFrame
# feature_names = ['Vek', 'RocniPrijem', 'CastkaUveru', 'DobaTrvani', 
#                 'KreditniSkore', 'DelkaZamestnani', 'MaPredchoziUvery']
# df = pd.DataFrame(X, columns=feature_names)
# df['StatusSelhani'] = y

# # Zobrazení dat
# print(df.head())
# print("\nMíra selhání:", df['StatusSelhani'].mean())

# # Vizualizace vztahu příznaků a cílové proměnné
# plt.figure(figsize=(15, 10))

# plt.subplot(2, 3, 1)
# sns.boxplot(x='StatusSelhani', y='RocniPrijem', data=df)
# plt.title('Roční příjem podle stavu selhání')

# plt.subplot(2, 3, 2)
# sns.boxplot(x='StatusSelhani', y='CastkaUveru', data=df)
# plt.title('Částka úvěru podle stavu selhání')

# plt.subplot(2, 3, 3)
# sns.boxplot(x='StatusSelhani', y='KreditniSkore', data=df)
# plt.title('Kreditní skóre podle stavu selhání')

# plt.subplot(2, 3, 4)
# sns.boxplot(x='StatusSelhani', y='DobaTrvani', data=df)
# plt.title('Doba trvání úvěru podle stavu selhání')

# plt.subplot(2, 3, 5)
# sns.boxplot(x='StatusSelhani', y='DelkaZamestnani', data=df)
# plt.title('Délka zaměstnání podle stavu selhání')

# plt.tight_layout()
# plt.show()

# # Rozdělení dat
# X = df.drop('StatusSelhani', axis=1)
# y = df['StatusSelhani']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Škálování příznaků
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # Vytvoření jednotlivých modelů
# log_reg = LogisticRegression(random_state=42)
# rf = RandomForestClassifier(n_estimators=100, random_state=42)
# gb = GradientBoostingClassifier(n_estimators=100, random_state=42)

# # Vytvoření ensemble modelu s hlasováním
# voting_clf = VotingClassifier(
#     estimators=[('lr', log_reg), ('rf', rf), ('gb', gb)],
#     voting='soft'  # Použití pravděpodobností pro hlasování
# )

# # Trénování všech modelů
# models = {
#     'Logistická regrese': log_reg,
#     'Random Forest': rf,
#     'Gradient Boosting': gb,
#     'Hlasovací klasifikátor': voting_clf
# }

# for name, model in models.items():
#     model.fit(X_train_scaled, y_train)
    
# # Vyhodnocení všech modelů
# plt.figure(figsize=(10, 8))

# for name, model in models.items():
#     y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
#     fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
#     roc_auc = auc(fpr, tpr)
#     plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

# plt.plot([0, 1], [0, 1], 'k--')
# plt.xlabel('Míra falešně pozitivních')
# plt.ylabel('Míra skutečně pozitivních')
# plt.title('Porovnání ROC křivek')
# plt.legend(loc='lower right')
# plt.grid(True, alpha=0.3)
# plt.show()

# # Metriky výkonu
# metrics_df = pd.DataFrame(columns=['Model', 'Přesnost', 'Preciznost', 'Úplnost', 'F1 skóre', 'ROC AUC'])

# for name, model in models.items():
#     y_pred = model.predict(X_test_scaled)
#     y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
#     acc = accuracy_score(y_test, y_pred)
#     prec = precision_score(y_test, y_pred)
#     rec = recall_score(y_test, y_pred)
#     f1 = f1_score(y_test, y_pred)
#     roc = roc_auc_score(y_test, y_pred_proba)
    
#     metrics_df = pd.concat([metrics_df, pd.DataFrame({
#         'Model': [name],
#         'Přesnost': [acc],
#         'Preciznost': [prec],
#         'Úplnost': [rec],
#         'F1 skóre': [f1],
#         'ROC AUC': [roc]
#     })], ignore_index=True)

# print("\nPorovnání výkonu modelů:")
# print(metrics_df)

# # Vizualizace matice záměn pro ensemble model
# y_pred_ensemble = voting_clf.predict(X_test_scaled)
# cm = confusion_matrix(y_test, y_pred_ensemble)

# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
# plt.xlabel('Predikované')
# plt.ylabel('Skutečné')
# plt.title('Matice záměn - Ensemble model')
# plt.show()

# # Důležitost příznaků z Random Forest
# feature_importances = pd.DataFrame({
#     'Příznak': X.columns,
#     'Důležitost': rf.feature_importances_
# }).sort_values('Důležitost', ascending=False)

# plt.figure(figsize=(10, 6))
# sns.barplot(x='Důležitost', y='Příznak', data=feature_importances)
# plt.title('Důležitost příznaků z Random Forest')
# plt.tight_layout()
# plt.show()

```
ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int32(0)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_curve, auc, roc_auc_score
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Vytvoření vyváženějšího syntetického datasetu pro nesplácení půjček
np.random.seed(42)
n_samples = 200

# Generování vlastností pro splácející klienty (přibližně 70% dat)
n_non_default = int(n_samples * 0.7)
X_non_default = np.zeros((n_non_default, 7))
X_non_default[:, 0] = np.random.normal(45, 10, n_non_default)  # Věk: průměr 45, std 10
X_non_default[:, 1] = np.random.normal(70000, 20000, n_non_default)  # Roční příjem: vyšší
X_non_default[:, 2] = np.random.normal(12000, 5000, n_non_default)  # Výše půjčky: nižší
X_non_default[:, 3] = np.random.uniform(1, 5, n_non_default)  # Doba trvání půjčky: kratší
X_non_default[:, 4] = np.random.normal(700, 70, n_non_default)  # Kreditní skóre: vyšší
X_non_default[:, 5] = np.random.poisson(5, n_non_default)  # Délka zaměstnání: delší
X_non_default[:, 6] = np.random.binomial(1, 0.8, n_non_default)  # Má předchozí půjčky (0 nebo 1)

# Generování vlastností pro nesplácející klienty (přibližně 30% dat)
n_default = n_samples - n_non_default
X_default = np.zeros((n_default, 7))
X_default[:, 0] = np.random.normal(35, 12, n_default)  # Věk: průměr 35, std 12
X_default[:, 1] = np.random.normal(45000, 18000, n_default)  # Roční příjem: nižší
X_default[:, 2] = np.random.normal(22000, 8000, n_default)  # Výše půjčky: vyšší
X_default[:, 3] = np.random.uniform(3, 7, n_default)  # Doba trvání půjčky: delší
X_default[:, 4] = np.random.normal(580, 80, n_default)  # Kreditní skóre: nižší
X_default[:, 5] = np.random.poisson(2, n_default)  # Délka zaměstnání: kratší
X_default[:, 6] = np.random.binomial(1, 0.5, n_default)  # Má předchozí půjčky (0 nebo 1)

# Spojení datasetů
X = np.vstack((X_non_default, X_default))
y = np.hstack((np.zeros(n_non_default), np.ones(n_default)))

# Zajištění rozumných rozsahů pro vlastnosti
X[:, 0] = np.clip(X[:, 0], 18, 70)  # Věk mezi 18 a 70
X[:, 1] = np.clip(X[:, 1], 20000, 150000)  # Příjem mezi 20 000 a 150 000
X[:, 2] = np.clip(X[:, 2], 5000, 50000)  # Výše půjčky mezi 5 000 a 50 000
X[:, 4] = np.clip(X[:, 4], 400, 850)  # Kreditní skóre mezi 400 a 850
X[:, 5] = np.clip(X[:, 5], 0, 20)  # Délka zaměstnání mezi 0 a 20 lety

# Promíchání dat
indices = np.random.permutation(n_samples)
X = X[indices]
y = y[indices]

# Vytvoření DataFrame
feature_names = ['Věk', 'RočníPříjem', 'VýšePůjčky', 'DobaSplácení', 
                'KreditníSkóre', 'DélkaZaměstnání', 'MáPředchozíPůjčky']
df = pd.DataFrame(X, columns=feature_names)
df['StavNesplácení'] = y

# Zobrazení dat
print(df.head())
print("\nMíra nesplácení:", df['StavNesplácení'].mean())
print("Počet nesplácených:", df['StavNesplácení'].sum())
print("Počet splácených:", (df['StavNesplácení'] == 0).sum())

# Vizualizace vztahů mezi vlastnostmi a cílovým atributem
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
sns.boxplot(x='StavNesplácení', y='RočníPříjem', data=df)
plt.title('Roční příjem podle stavu nesplácení')

plt.subplot(2, 3, 2)
sns.boxplot(x='StavNesplácení', y='VýšePůjčky', data=df)
plt.title('Výše půjčky podle stavu nesplácení')

plt.subplot(2, 3, 3)
sns.boxplot(x='StavNesplácení', y='KreditníSkóre', data=df)
plt.title('Kreditní skóre podle stavu nesplácení')

plt.subplot(2, 3, 4)
sns.boxplot(x='StavNesplácení', y='DobaSplácení', data=df)
plt.title('Doba splácení podle stavu nesplácení')

plt.subplot(2, 3, 5)
sns.boxplot(x='StavNesplácení', y='DélkaZaměstnání', data=df)
plt.title('Délka zaměstnání podle stavu nesplácení')

plt.tight_layout()
plt.show()

# Rozdělení dat se stratifikací pro zajištění zastoupení obou tříd
X = df.drop('StavNesplácení', axis=1)
y = df['StavNesplácení']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Ověření distribuce tříd v trénovací a testovací sadě
print("\nDistribuce tříd v trénovací sadě:")
print(y_train.value_counts())
print("\nDistribuce tříd v testovací sadě:")
print(y_test.value_counts())

# Škálování vlastností
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Vytvoření jednotlivých modelů
log_reg = LogisticRegression(random_state=42, max_iter=1000)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)

# Vytvoření hlasovacího ensemble modelu
voting_clf = VotingClassifier(
    estimators=[('lr', log_reg), ('rf', rf), ('gb', gb)],
    voting='soft'  # Použití pravděpodobností pro hlasování
)

# Trénování všech modelů
models = {
    'Logistická regrese': log_reg,
    'Random Forest': rf,
    'Gradient Boosting': gb,
    'Hlasovací klasifikátor': voting_clf
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    
# Vyhodnocení všech modelů
plt.figure(figsize=(10, 8))

for name, model in models.items():
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Míra falešně pozitivních')
plt.ylabel('Míra správně pozitivních')
plt.title('Srovnání ROC křivek')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

# Metriky výkonu: oprava FutureWarning vytvořením seznamu a pak DataFrame najednou
metrics_list = []

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_pred_proba)
    
    metrics_list.append({
        'Model': name,
        'Přesnost': acc,
        'Precize': prec,
        'Úplnost': rec,
        'F1 skóre': f1,
        'ROC AUC': roc
    })

metrics_df = pd.DataFrame(metrics_list)

print("\nPorovnání výkonu modelů:")
print(metrics_df)

# Vizualizace matice záměn pro ensemble model
y_pred_ensemble = voting_clf.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_ensemble)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predikováno')
plt.ylabel('Skutečnost')
plt.title('Matice záměn - Ensemble model')
plt.show()

# Důležitost vlastností z Random Forest modelu
feature_importances = pd.DataFrame({
    'Vlastnost': X.columns,
    'Důležitost': rf.feature_importances_
}).sort_values('Důležitost', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Důležitost', y='Vlastnost', data=feature_importances)
plt.title('Důležitost vlastností z Random Forest modelu')
plt.tight_layout()
plt.show()

## 6. Detekce anomálií

### Popis
Detekce anomálií identifikuje neobvyklé vzory nebo odlehlé hodnoty, které neodpovídají očekávanému chování. Pomáhá detekovat podvody, chyby, defekty nebo jakékoli neobvyklé aktivity v datech.

**Aplikace:** Detekce podvodů, síťové intruse, výrobní defekty, monitoring zdraví systémů

### Příklad: Anomálie v transakcích kreditních karet

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.preprocessing import StandardScaler
# from sklearn.ensemble import IsolationForest
# from sklearn.neighbors import LocalOutlierFactor
# from sklearn.svm import OneClassSVM
# from sklearn.metrics import precision_recall_fscore_support

# # Vytvoření syntetického datasetu transakcí kreditních karet
# np.random.seed(42)
# n_samples = 200
# n_outliers = 20
# n_inliers = n_samples - n_outliers

# # Generování normálních transakčních vzorů
# X_normal = np.zeros((n_inliers, 4))
# X_normal[:, 0] = np.random.normal(100, 50, n_inliers)  # Částka transakce: průměr $100, std $50
# X_normal[:, 0] = np.abs(X_normal[:, 0])  # Zajištění kladných částek
# X_normal[:, 1] = np.random.normal(5, 3, n_inliers)  # Frekvence transakcí (za týden)
# X_normal[:, 1] = np.abs(X_normal[:, 1])
# X_normal[:, 2] = np.random.normal(15, 7, n_inliers)  # Vzdálenost od domova (míle)
# X_normal[:, 2] = np.abs(X_normal[:, 2])
# X_normal[:, 3] = np.random.normal(0.3, 0.2, n_inliers)  # Čas dne (normalizovaný 0-1: 0=půlnoc, 1=konec dne)
# X_normal[:, 3] = np.clip(X_normal[:, 3], 0, 1)  # Zajištění hodnot mezi 0 a 1

# # Generování anomálních transakcí
# X_anomalous = np.zeros((n_outliers, 4))
# # Náhodné vytvoření různých typů anomálií
# anomaly_types = np.random.choice(3, n_outliers)

# for i in range(n_outliers):
#     if anomaly_types[i] == 0:  # Anomálie velké částky
#         X_anomalous[i, 0] = np.random.uniform(300, 1000)
#         X_anomalous[i, 1] = np.random.normal(5, 3)
#         X_anomalous[i, 2] = np.random.normal(15, 7)
#         X_anomalous[i, 3] = np.random.normal(0.3, 0.2)
#     elif anomaly_types[i] == 1:  # Anomálie neobvyklého místa
#         X_anomalous[i, 0] = np.random.normal(100, 50)
#         X_anomalous[i, 1] = np.random.normal(5, 3)
#         X_anomalous[i, 2] = np.random.uniform(100, 1000)
#         X_anomalous[i, 3] = np.random.normal(0.3, 0.2)
#     else:  # Anomálie neobvyklého času
#         X_anomalous[i, 0] = np.random.normal(100, 50)
#         X_anomalous[i, 1] = np.random.normal(5, 3)
#         X_anomalous[i, 2] = np.random.normal(15, 7)
#         X_anomalous[i, 3] = np.random.uniform(0.8, 1)  # Velmi pozdě v noci

# X_anomalous = np.abs(X_anomalous)  # Zajištění kladných hodnot
# X_anomalous[:, 3] = np.clip(X_anomalous[:, 3], 0, 1)

# # Kombinace normálních a anomálních dat
# X = np.vstack([X_normal, X_anomalous])
# y_true = np.zeros(n_samples)
# y_true[n_inliers:] = 1  # Označení anomálií jako 1

# # Vytvoření DataFrame
# feature_names = ['CastkaTransakce', 'TydenFrekvence', 'VzdalenostOdDomova', 'CasDne']
# df = pd.DataFrame(X, columns=feature_names)
# df['JeAnomalie'] = y_true

# # Zobrazení dat
# print(df.head())
# print("\nDistribuce anomálií:", df['JeAnomalie'].value_counts())

# # Vizualizace distribuce dat
# plt.figure(figsize=(14, 10))

# plt.subplot(2, 2, 1)
# sns.histplot(data=df, x='CastkaTransakce', hue='JeAnomalie', bins=20, kde=True)
# plt.title('Distribuce částek transakcí')

# plt.subplot(2, 2, 2)
# sns.histplot(data=df, x='VzdalenostOdDomova', hue='JeAnomalie', bins=20, kde=True)
# plt.title('Distribuce vzdáleností od domova')

# plt.subplot(2, 2, 3)
# sns.scatterplot(data=df, x='CastkaTransakce', y='VzdalenostOdDomova', hue='JeAnomalie', alpha=0.7)
# plt.title('Částka transakce vs. Vzdálenost')

# plt.subplot(2, 2, 4)
# sns.scatterplot(data=df, x='CasDne', y='CastkaTransakce', hue='JeAnomalie', alpha=0.7)
# plt.title('Čas dne vs. Částka transakce')

# plt.tight_layout()
# plt.show()

# # Škálování příznaků
# X = df.drop('JeAnomalie', axis=1).values
# y_true = df['JeAnomalie'].values
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# # Aplikace různých algoritmů detekce anomálií
# # Isolation Forest
# isolation_forest = IsolationForest(contamination=n_outliers/n_samples, random_state=42)
# y_pred_if = isolation_forest.fit_predict(X_scaled)
# y_pred_if = np.where(y_pred_if == -1, 1, 0)  # Převod na binární (1 pro anomálii)

# # Local Outlier Factor
# lof = LocalOutlierFactor(n_neighbors=20, contamination=n_outliers/n_samples)
# y_pred_lof = lof.fit_predict(X_scaled)
# y_pred_lof = np.where(y_pred_lof == -1, 1, 0)  # Převod na binární (1 pro anomálii)

# # One-Class SVM
# ocsvm = OneClassSVM(nu=n_outliers/n_samples, kernel='rbf', gamma='scale')
# y_pred_ocsvm = ocsvm.fit_predict(X_scaled)
# y_pred_ocsvm = np.where(y_pred_ocsvm == -1, 1, 0)  # Převod na binární (1 pro anomálii)

# # Přidání predikcí do DataFrame pro vizualizaci
# df['IF_Predikce'] = y_pred_if
# df['LOF_Predikce'] = y_pred_lof
# df['OCSVM_Predikce'] = y_pred_ocsvm

# # Vyhodnocení modelů
# methods = ['Isolation Forest', 'Local Outlier Factor', 'One-Class SVM']
# predictions = [y_pred_if, y_pred_lof, y_pred_ocsvm]

# eval_metrics = pd.DataFrame(columns=['Metoda', 'Preciznost', 'Úplnost', 'F1-Skóre'])

# for method, y_pred in zip(methods, predictions):
#     precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
#     eval_metrics = pd.concat([eval_metrics, pd.DataFrame({
#         'Metoda': [method],
#         'Preciznost': [precision],
#         'Úplnost': [recall],
#         'F1-Skóre': [f1]
#     })], ignore_index=True)

# print("\nVyhodnocení modelů:")
# print(eval_metrics)

# # Vizualizace predikcí nejlepší metody (podle F1)
# best_method_idx = eval_metrics['F1-Skóre'].idxmax()
# best_method = methods[best_method_idx]
# best_predictions = predictions[best_method_idx]

# plt.figure(figsize=(14, 8))

# plt.subplot(1, 2, 1)
# plt.scatter(df['CastkaTransakce'], df['VzdalenostOdDomova'], c=df['JeAnomalie'], 
#             cmap='viridis', edgecolor='k', s=50, alpha=0.7)
# plt.title('Skutečné anomálie')
# plt.xlabel('Částka transakce')
# plt.ylabel('Vzdálenost od domova')
# plt.colorbar(label='Je anomálie')

# plt.subplot(1, 2, 2)
# plt.scatter(df['CastkaTransakce'], df['VzdalenostOdDomova'], c=best_predictions, 
#             cmap='viridis', edgecolor='k', s=50, alpha=0.7)
# plt.title(f'Predikované anomálie ({best_method})')
# plt.xlabel('Částka transakce')
# plt.ylabel('Vzdálenost od domova')
# plt.colorbar(label='Predikovaná anomálie')

# plt.tight_layout()
# plt.show()

# # Matice záměn pro všechny metody
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# for i, (method, y_pred) in enumerate(zip(methods, predictions)):
#     cm = pd.crosstab(y_true, y_pred, rownames=['Skutečné'], colnames=['Predikované'])
#     sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
#     axes[i].set_title(f'Matice záměn - {method}')

# plt.tight_layout()
# plt.show()

# # Vizualizace skóre anomálií (pro Isolation Forest)
# anomaly_scores = isolation_forest.score_samples(X_scaled)
# df['SkoreAnomalie'] = -anomaly_scores  # Negace pro vyšší skóre = více anomální

# plt.figure(figsize=(10, 6))
# plt.scatter(range(len(df)), df['SkoreAnomalie'], c=df['JeAnomalie'], cmap='coolwarm', alpha=0.7)
# plt.axhline(y=isolation_forest.threshold_, color='r', linestyle='--', label='Práh')
# plt.title('Skóre anomálií z Isolation Forest')
# plt.xlabel('Index datového bodu')
# plt.ylabel('Skóre anomálie')
# plt.colorbar(label='Skutečná anomálie')
# plt.legend()
# plt.tight_layout()
# plt.show()

```
FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

AttributeError: 'IsolationForest' object has no attribute 'threshold_'
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import precision_recall_fscore_support

# Vytvoření syntetického datasetu transakcí kreditních karet
np.random.seed(42)
n_samples = 200
n_outliers = 20
n_inliers = n_samples - n_outliers

# Generování normálních vzorů transakcí
X_normal = np.zeros((n_inliers, 4))
X_normal[:, 0] = np.abs(np.random.normal(100, 50, n_inliers))  # Částka transakce
X_normal[:, 1] = np.abs(np.random.normal(5, 3, n_inliers))     # Týdenní frekvence
X_normal[:, 2] = np.abs(np.random.normal(15, 7, n_inliers))    # Vzdálenost od domova
X_normal[:, 3] = np.clip(np.random.normal(0.3, 0.2, n_inliers), 0, 1)  # Čas dne

# Generování anomálních transakcí
X_anomalous = np.zeros((n_outliers, 4))
anomaly_types = np.random.choice(3, n_outliers)

for i in range(n_outliers):
    if anomaly_types[i] == 0:  # Vysoká částka
        X_anomalous[i] = [np.random.uniform(300, 1000),
                          np.random.normal(5, 3),
                          np.random.normal(15, 7),
                          np.random.normal(0.3, 0.2)]
    elif anomaly_types[i] == 1:  # Neobvyklá lokace
        X_anomalous[i] = [np.random.normal(100, 50),
                          np.random.normal(5, 3),
                          np.random.uniform(100, 1000),
                          np.random.normal(0.3, 0.2)]
    else:  # Neobvyklý čas
        X_anomalous[i] = [np.random.normal(100, 50),
                          np.random.normal(5, 3),
                          np.random.normal(15, 7),
                          np.random.uniform(0.8, 1)]

X_anomalous = np.abs(X_anomalous)
X_anomalous[:, 3] = np.clip(X_anomalous[:, 3], 0, 1)

# Kombinace a vytvoření DataFrame
X = np.vstack([X_normal, X_anomalous])
y_true = np.zeros(n_samples)
y_true[n_inliers:] = 1  # Anomálie označeny jako 1

nazvy_priznaku = ['ČástkaTransakce', 'TýdenníFrekvence', 'VzdálenostOdDomova', 'ČasDne']
df = pd.DataFrame(X, columns=nazvy_priznaku)
df['JeAnomálie'] = y_true

print(df.head())
print("\nRozdělení anomálií:", df['JeAnomálie'].value_counts())

# Vizualizace
plt.figure(figsize=(14, 10))
plt.subplot(2, 2, 1)
sns.histplot(data=df, x='ČástkaTransakce', hue='JeAnomálie', bins=20, kde=True)
plt.title('Distribuce částky transakce')

plt.subplot(2, 2, 2)
sns.histplot(data=df, x='VzdálenostOdDomova', hue='JeAnomálie', bins=20, kde=True)
plt.title('Distribuce vzdálenosti od domova')

plt.subplot(2, 2, 3)
sns.scatterplot(data=df, x='ČástkaTransakce', y='VzdálenostOdDomova', hue='JeAnomálie', alpha=0.7)
plt.title('Částka transakce vs. Vzdálenost')

plt.subplot(2, 2, 4)
sns.scatterplot(data=df, x='ČasDne', y='ČástkaTransakce', hue='JeAnomálie', alpha=0.7)
plt.title('Čas dne vs. Částka transakce')

plt.tight_layout()
plt.show()

# Škálování příznaků
X = df.drop('JeAnomálie', axis=1).values
y_true = df['JeAnomálie'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Isolation Forest
isolation_forest = IsolationForest(contamination=n_outliers/n_samples, random_state=42)
y_pred_if = isolation_forest.fit_predict(X_scaled)
y_pred_if = np.where(y_pred_if == -1, 1, 0)

# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=20, contamination=n_outliers/n_samples)
y_pred_lof = lof.fit_predict(X_scaled)
y_pred_lof = np.where(y_pred_lof == -1, 1, 0)

# One-Class SVM
ocsvm = OneClassSVM(nu=n_outliers/n_samples, kernel='rbf', gamma='scale')
y_pred_ocsvm = ocsvm.fit_predict(X_scaled)
y_pred_ocsvm = np.where(y_pred_ocsvm == -1, 1, 0)

# Přidání predikcí do DataFrame
df['IF_Predikce'] = y_pred_if
df['LOF_Predikce'] = y_pred_lof
df['OCSVM_Predikce'] = y_pred_ocsvm

# Vyhodnocení
metody = ['Isolation Forest', 'Local Outlier Factor', 'One-Class SVM']
predikce = [y_pred_if, y_pred_lof, y_pred_ocsvm]

eval_metrics = pd.DataFrame(columns=['Metoda', 'Preciznost', 'Recall', 'F1-Score'])

for metoda, y_pred in zip(metody, predikce):
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
    eval_metrics.loc[len(eval_metrics)] = [metoda, precision, recall, f1]

print("\nVyhodnocení modelů:")
print(eval_metrics)

# Vizualizace predikcí
best_method_idx = eval_metrics['F1-Score'].idxmax()
best_method = metody[best_method_idx]
best_predictions = predikce[best_method_idx]

plt.figure(figsize=(14, 8))

plt.subplot(1, 2, 1)
plt.scatter(df['ČástkaTransakce'], df['VzdálenostOdDomova'], c=df['JeAnomálie'], 
            cmap='viridis', edgecolor='k', s=50, alpha=0.7)
plt.title('Skutečné anomálie')
plt.xlabel('Částka transakce')
plt.ylabel('Vzdálenost od domova')
plt.colorbar(label='Je anomálie')

plt.subplot(1, 2, 2)
plt.scatter(df['ČástkaTransakce'], df['VzdálenostOdDomova'], c=best_predictions, 
            cmap='viridis', edgecolor='k', s=50, alpha=0.7)
plt.title(f'Predikované anomálie ({best_method})')
plt.xlabel('Částka transakce')
plt.ylabel('Vzdálenost od domova')
plt.colorbar(label='Predikovaná anomálie')

plt.tight_layout()
plt.show()

# Matice záměn
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (metoda, y_pred) in enumerate(zip(metody, predikce)):
    cm = pd.crosstab(y_true, y_pred, rownames=['Skutečnost'], colnames=['Predikce'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f'Matice záměn - {metoda}')

plt.tight_layout()
plt.show()

# Skóre anomálií z Isolation Forest
anomaly_scores = isolation_forest.score_samples(X_scaled)
df['SkóreAnomálie'] = -anomaly_scores  # Vyšší skóre = více anomální

# Definice prahu pro vizualizaci (např. 95. percentil)
visual_threshold = np.percentile(df['SkóreAnomálie'], 95)

plt.figure(figsize=(10, 6))
plt.scatter(range(len(df)), df['SkóreAnomálie'], c=df['JeAnomálie'], cmap='coolwarm', alpha=0.7)
plt.axhline(y=visual_threshold, color='r', linestyle='--', label='95. percentil práh')
plt.title('Skóre anomálií z Isolation Forest')
plt.xlabel('Index datového bodu')
plt.ylabel('Skóre anomálie')
plt.colorbar(label='Skutečná anomálie')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Analýza časových řad

### Popis
Analýza časových řad zkoumá datové body shromážděné nebo indexované v průběhu času za účelem extrakce smysluplných vzorů a trendů. Pomáhá při předpovídání, pochopení časových vzorů a analýze časově závislých procesů.

**Aplikace:** Predikce akciového trhu, prognóza prodejů, předpověď počasí, analýza ekonomických ukazatelů

### Příklad: Prognóza měsíčních prodejů

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from statsmodels.tsa.seasonal import seasonal_decompose
# from statsmodels.tsa.arima.model import ARIMA
# from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# from datetime import datetime, timedelta

# # Vytvoření syntetických dat časových řad
# np.random.seed(42)
# n_samples = 200  # ~16,5 let měsíčních dat

# # Vytvoření datového rozsahu
# start_date = datetime(2007, 1, 1)
# dates = [start_date + timedelta(days=30*i) for i in range(n_samples)]

# # Trendová složka
# trend = np.linspace(100, 250, n_samples)

# # Sezónní složka (roční sezónnost s vrcholem v prosinci)
# period = 12  # měsíční data, perioda = 12 měsíců
# seasonal = 40 * np.sin(2 * np.pi * np.arange(n_samples) / period)

# # Speciální události (např. skoky v prodeji během svátků každý prosinec)
# special_events = np.zeros(n_samples)
# for i in range(n_samples):
#     month = dates[i].month
#     if month == 12:  # Prosinec
#         special_events[i] = 30 + np.random.normal(0, 5)
#     elif month == 1:  # Leden (pokles po svátcích)
#         special_events[i] = -15 + np.random.normal(0, 3)

# # Náhodný šum
# noise = np.random.normal(0, 15, n_samples)

# # Kombinace složek
# sales = trend + seasonal + special_events + noise

# # Vytvoření DataFrame
# df = pd.DataFrame({
#     'Date': dates,
#     'Sales': sales
# })
# df.set_index('Date', inplace=True)

# # Zobrazení dat
# print(df.head())
# print("\nStatistika datasetu:")
# print(df.describe())

# # Vykreslení časové řady
# plt.figure(figsize=(14, 7))
# plt.plot(df.index, df['Sales'], marker='o', markersize=3)
# plt.title('Měsíční data prodejů')
# plt.xlabel('Datum')
# plt.ylabel('Prodeje')
# plt.grid(True)
# plt.tight_layout()
# plt.show()

# # Dekompozice časové řady
# decomposition = seasonal_decompose(df['Sales'], model='additive', period=12)

# plt.figure(figsize=(14, 10))

# plt.subplot(4, 1, 1)
# plt.plot(df.index, df['Sales'])
# plt.title('Původní časová řada')
# plt.grid(True)

# plt.subplot(4, 1, 2)
# plt.plot(decomposition.trend)
# plt.title('Trendová složka')
# plt.grid(True)

# plt.subplot(4, 1, 3)
# plt.plot(decomposition.seasonal)
# plt.title('Sezónní složka')
# plt.grid(True)

# plt.subplot(4, 1, 4)
# plt.plot(decomposition.resid)
# plt.title('Rezidua (náhodný šum)')
# plt.grid(True)

# plt.tight_layout()
# plt.show()

# # Výpočet a vykreslení autokorelace a parciální autokorelace
# plt.figure(figsize=(14, 7))

# plt.subplot(2, 1, 1)
# plot_acf(df['Sales'], ax=plt.gca(), lags=36)
# plt.title('Autokorelační funkce (ACF)')

# plt.subplot(2, 1, 2)
# plot_pacf(df['Sales'], ax=plt.gca(), lags=36)
# plt.title('Parciální autokorelační funkce (PACF)')

# plt.tight_layout()
# plt.show()

# # Měsíční tempo růstu
# df['Growth'] = df['Sales'].pct_change() * 100

# plt.figure(figsize=(14, 7))
# plt.bar(df.index[1:], df['Growth'][1:], width=20)
# plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
# plt.title('Měsíční tempo růstu prodejů (%)')
# plt.xlabel('Datum')
# plt.ylabel('Tempo růstu (%)')
# plt.grid(True, axis='y')
# plt.tight_layout()
# plt.show()

# # Sezónní graf
# plt.figure(figsize=(14, 7))
# year_groups = df.groupby(df.index.month)
# for month, month_data in year_groups:
#     plt.plot(month_data.index.year, month_data['Sales'], label=f'Měsíc {month}')

# plt.title('Prodeje podle měsíců napříč roky')
# plt.xlabel('Rok')
# plt.ylabel('Prodeje')
# plt.grid(True)
# plt.legend(title='Měsíc', bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.tight_layout()
# plt.show()

# # Rozdělení dat na trénovací a testovací část
# train_size = int(len(df) * 0.8)
# train_data = df.iloc[:train_size]
# test_data = df.iloc[train_size:]

# plt.figure(figsize=(14, 7))
# plt.plot(train_data.index, train_data['Sales'], label='Trénovací data')
# plt.plot(test_data.index, test_data['Sales'], label='Testovací data')
# plt.title('Rozdělení na trénovací a testovací data')
# plt.xlabel('Datum')
# plt.ylabel('Prodeje')
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()

# # Vytvoření ARIMA modelu
# # Na základě analýzy ACF a PACF vybereme vhodné hodnoty p, d, q
# # Pro tento příklad použijeme ARIMA(2,1,2) - zjednodušené pro ukázku
# p, d, q = 2, 1, 2

# model = ARIMA(train_data['Sales'], order=(p, d, q))
# results = model.fit()
# print(results.summary())

# # Předpověď
# forecast_steps = len(test_data)
# forecast = results.forecast(steps=forecast_steps)
# forecast_index = test_data.index

# # Vykreslení předpovědi vs. skutečné hodnoty
# plt.figure(figsize=(14, 7))
# plt.plot(train_data.index, train_data['Sales'], label='Trénovací data')
# plt.plot(test_data.index, test_data['Sales'], label='Skutečná testovací data')
# plt.plot(forecast_index, forecast, label='Předpověď', color='red')
# plt.title(f'Předpověď prodejů pomocí ARIMA({p},{d},{q})')
# plt.xlabel('Datum')
# plt.ylabel('Prodeje')
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()

# # Vyhodnocení předpovědi
# mse = mean_squared_error(test_data['Sales'], forecast)
# rmse = np.sqrt(mse)
# mae = mean_absolute_error(test_data['Sales'], forecast)
# r2 = r2_score(test_data['Sales'], forecast)

# print("\nHodnocení předpovědi:")
# print(f"Střední kvadratická chyba (MSE): {mse:.2f}")
# print(f"Odmocnina střední kvadratické chyby (RMSE): {rmse:.2f}")
# print(f"Střední absolutní chyba (MAE): {mae:.2f}")
# print(f"Skóre R²: {r2:.2f}")

# # Chyby předpovědi
# errors = test_data['Sales'].values - forecast
# plt.figure(figsize=(14, 7))
# plt.bar(test_data.index, errors)
# plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
# plt.title('Chyby předpovědi')
# plt.xlabel('Datum')
# plt.ylabel('Chyba (Skutečná - Předpověď)')
# plt.grid(True, axis='y')
# plt.tight_layout()
# plt.show()

# # Budoucí předpověď (příštích 12 měsíců)
# future_steps = 12
# extended_forecast = results.forecast(steps=forecast_steps + future_steps)

# # Vytvoření budoucích datumů
# last_date = df.index[-1]
# future_dates = [last_date + timedelta(days=30*i) for i in range(1, future_steps + 1)]
# future_index = pd.DatetimeIndex(future_dates)

# # Kombinace všech dat pro vykreslení
# plt.figure(figsize=(14, 7))
# plt.plot(df.index, df['Sales'], label='Historická data')
# plt.plot(pd.DatetimeIndex(list(forecast_index) + list(future_index)), 
#          extended_forecast[-future_steps:], 
#          label='Budoucí předpověď', color='red')
# plt.axvline(x=test_data.index[0], color='black', linestyle='--', 
#             label='Rozdělení trénovací-testovací data')
# plt.axvline(x=df.index[-1], color='green', linestyle='--', 
#             label='Začátek předpovědi')
# plt.title('Předpověď prodejů - Historická a budoucí')
# plt.xlabel('Datum')
# plt.ylabel('Prodeje')
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()

```
ValueWarning: No frequency information was provided, so inferred frequency 30D will be used.

ValueError: x and y must have same first dimension, but have shapes (52,) and (12,)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from datetime import datetime, timedelta

# Vytvoření syntetických dat časové řady
np.random.seed(42)
n_samples = 200  # ~16.5 let měsíčních dat

# Vytvoření rozsahu dat s explicitní frekvencí
start_date = datetime(2007, 1, 1)
dates = pd.date_range(start=start_date, periods=n_samples, freq='30D')

# Trendová složka
trend = np.linspace(100, 250, n_samples)

# Sezónní složka (roční sezónnost s vrcholem v prosinci)
period = 12
seasonal = 40 * np.sin(2 * np.pi * np.arange(n_samples) / period)

# Speciální události (např. vánoční špičky prodejů každý prosinec)
special_events = np.zeros(n_samples)
for i in range(n_samples):
    month = dates[i].month
    if month == 12:
        special_events[i] = 30 + np.random.normal(0, 5)
    elif month == 1:
        special_events[i] = -15 + np.random.normal(0, 3)

# Náhodný šum
noise = np.random.normal(0, 15, n_samples)

# Kombinace složek
sales = trend + seasonal + special_events + noise

# Vytvoření DataFrame
df = pd.DataFrame({'Prodeje': sales}, index=dates)

# Zobrazení dat
print(df.head())
print("\nStatistiky datasetu:")
print(df.describe())

# Graf časové řady
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Prodeje'], marker='o', markersize=3)
plt.title('Měsíční data prodejů')
plt.xlabel('Datum')
plt.ylabel('Prodeje')
plt.grid(True)
plt.tight_layout()
plt.show()

# Dekompozice časové řady
decomposition = seasonal_decompose(df['Prodeje'], model='additive', period=12)

plt.figure(figsize=(14, 10))
plt.subplot(4, 1, 1)
plt.plot(df.index, df['Prodeje'])
plt.title('Původní časová řada')
plt.grid(True)

plt.subplot(4, 1, 2)
plt.plot(decomposition.trend)
plt.title('Trendová složka')
plt.grid(True)

plt.subplot(4, 1, 3)
plt.plot(decomposition.seasonal)
plt.title('Sezónní složka')
plt.grid(True)

plt.subplot(4, 1, 4)
plt.plot(decomposition.resid)
plt.title('Rezidua (náhodný šum)')
plt.grid(True)

plt.tight_layout()
plt.show()

# ACF a PACF grafy
plt.figure(figsize=(14, 7))
plt.subplot(2, 1, 1)
plot_acf(df['Prodeje'], ax=plt.gca(), lags=36)
plt.title('Autokorelační funkce (ACF)')

plt.subplot(2, 1, 2)
plot_pacf(df['Prodeje'], ax=plt.gca(), lags=36)
plt.title('Parciální autokorelační funkce (PACF)')

plt.tight_layout()
plt.show()

# Měsíční míra růstu
df['Růst'] = df['Prodeje'].pct_change() * 100

plt.figure(figsize=(14, 7))
plt.bar(df.index[1:], df['Růst'][1:], width=20)
plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
plt.title('Měsíční míra růstu prodejů (%)')
plt.xlabel('Datum')
plt.ylabel('Míra růstu (%)')
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

# Sezónní graf
plt.figure(figsize=(14, 7))
year_groups = df.groupby(df.index.month)
for month, month_data in year_groups:
    plt.plot(month_data.index.year, month_data['Prodeje'], label=f'Měsíc {month}')

plt.title('Prodeje podle měsíce v průběhu let')
plt.xlabel('Rok')
plt.ylabel('Prodeje')
plt.grid(True)
plt.legend(title='Měsíc', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Rozdělení na trénovací a testovací data
train_size = int(len(df) * 0.8)
train_data = df.iloc[:train_size]
test_data = df.iloc[train_size:]

# Zajištění zachování frekvence
train_data.index.freq = '30D'
test_data.index.freq = '30D'

plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data['Prodeje'], label='Trénovací data')
plt.plot(test_data.index, test_data['Prodeje'], label='Testovací data')
plt.title('Rozdělení trénovacích a testovacích dat')
plt.xlabel('Datum')
plt.ylabel('Prodeje')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# ARIMA model
p, d, q = 2, 1, 2
model = ARIMA(train_data['Prodeje'], order=(p, d, q))
results = model.fit()
print(results.summary())

# Prognóza
forecast_steps = len(test_data)
forecast = results.forecast(steps=forecast_steps)
forecast_index = test_data.index

# Graf prognózy vs skutečnost
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data['Prodeje'], label='Trénovací data')
plt.plot(test_data.index, test_data['Prodeje'], label='Skutečná testovací data')
plt.plot(forecast_index, forecast, label='Prognóza', color='red')
plt.title(f'Prognóza prodejů s ARIMA({p},{d},{q})')
plt.xlabel('Datum')
plt.ylabel('Prodeje')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Vyhodnocení prognózy
mse = mean_squared_error(test_data['Prodeje'], forecast)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_data['Prodeje'], forecast)
r2 = r2_score(test_data['Prodeje'], forecast)

print("\nVyhodnocení prognózy:")
print(f"Střední kvadratická chyba (MSE): {mse:.2f}")
print(f"Odmocnina střední kvadratické chyby (RMSE): {rmse:.2f}")
print(f"Průměrná absolutní chyba (MAE): {mae:.2f}")
print(f"R² skóre: {r2:.2f}")

# Chyby prognózy
errors = test_data['Prodeje'].values - forecast
plt.figure(figsize=(14, 7))
plt.bar(test_data.index, errors)
plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
plt.title('Chyby prognózy')
plt.xlabel('Datum')
plt.ylabel('Chyba (Skutečnost - Prognóza)')
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

# Budoucí prognóza (dalších 12 měsíců)
future_steps = 12
extended_forecast = results.forecast(steps=forecast_steps + future_steps)

# Vygenerování úplného indexu prognózy: test + budoucnost
full_index = pd.date_range(start=forecast_index[0], periods=forecast_steps + future_steps, freq='30D')

# Graf úplné prognózy
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Prodeje'], label='Historická data')
plt.plot(full_index, extended_forecast, label='Prognóza (Test + Budoucnost)', color='red')
plt.axvline(x=test_data.index[0], color='black', linestyle='--', label='Rozdělení trénování-test')
plt.axvline(x=df.index[-1], color='green', linestyle='--', label='Začátek prognózy')
plt.title('Prognóza prodejů - Historické a budoucí')
plt.xlabel('Datum')
plt.ylabel('Prodeje')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Semi-supervizované učení

### Popis
Semi-supervizované učení využívá jak označená, tak neoznačená data pro trénování. Je cenné, když jsou označená data vzácná nebo drahá na získání, zatímco neoznačená data jsou dostupná ve velkém množství.

**Aplikace:** Klasifikace textu, rozpoznávání obrazů, rozpoznávání řeči, lékařská diagnostika

### Příklad: Šíření značek pomocí datasetu Two Moons

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.semi_supervised import LabelSpreading
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons

# Nastavení seedu pro reprodukovatelnost
np.random.seed(42)

# Generování syntetických dat - dataset dvou půlměsíců
X, y_true = make_moons(n_samples=200, noise=0.2, random_state=42)

# Vytvoření dataframe pro snadnější manipulaci
df = pd.DataFrame({
    'x1': X[:, 0],
    'x2': X[:, 1],
    'label': y_true
})

# Vizualizace skutečných značek
plt.figure(figsize=(10, 6))
plt.scatter(df['x1'], df['x2'], c=df['label'], cmap='viridis', 
           edgecolor='k', s=80, alpha=0.7)
plt.title('Původní dataset se skutečnými značkami')
plt.xlabel('Vlastnost 1')
plt.ylabel('Vlastnost 2')
plt.colorbar(label='Třída')
plt.grid(True, alpha=0.3)
plt.show()

# Rozdělení datasetu na označené a neoznačené části
# Použijeme pouze 10 % dat jako označené příklady
labeled_idx = []
for label in np.unique(y_true):
    idx = np.where(y_true == label)[0]
    labeled_idx.extend(idx[:10])  # Vezmeme pouze 10 vzorků na třídu

# Vytvoříme kopii skutečných značek a většinu označíme jako neznámé (-1)
y_semi = np.full(y_true.shape, -1)  # -1 představuje neoznačená data
y_semi[labeled_idx] = y_true[labeled_idx]

# Vizualizace označených vs. neoznačených dat
plt.figure(figsize=(10, 6))
# Vykreslení neoznačených bodů
mask_unlabeled = y_semi == -1
plt.scatter(X[mask_unlabeled, 0], X[mask_unlabeled, 1], 
           c='gray', marker='o', s=80, alpha=0.5, label='Neoznačené')
# Vykreslení označených bodů
mask_labeled = y_semi != -1
plt.scatter(X[mask_labeled, 0], X[mask_labeled, 1], 
           c=y_semi[mask_labeled], cmap='viridis', marker='*', 
           s=150, edgecolor='k', label='Označené')
plt.title('Dataset pro semi-supervizované učení')
plt.xlabel('Vlastnost 1')
plt.ylabel('Vlastnost 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Aplikace semi-supervizovaného učení Label Spreading
model = LabelSpreading(kernel='knn', n_neighbors=7, alpha=0.2)
model.fit(X, y_semi)

# Získání predikcí
y_pred = model.predict(X)

# Výpočet přesnosti na původně neoznačených datech
unlabeled_idx = np.where(y_semi == -1)[0]
accuracy_unlabeled = accuracy_score(y_true[unlabeled_idx], y_pred[unlabeled_idx])

# Vizualizace výsledků
plt.figure(figsize=(15, 5))

# Původní skutečné značky
plt.subplot(131)
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', 
           edgecolor='k', s=80, alpha=0.7)
plt.title('Skutečné značky')
plt.xlabel('Vlastnost 1')
plt.ylabel('Vlastnost 2')
plt.grid(True, alpha=0.3)

# Semi-supervizovaný vstup (označená + neoznačená data)
plt.subplot(132)
plt.scatter(X[mask_unlabeled, 0], X[mask_unlabeled, 1], 
           c='gray', s=80, alpha=0.5)
plt.scatter(X[mask_labeled, 0], X[mask_labeled, 1], 
           c=y_semi[mask_labeled], cmap='viridis', marker='*', 
           s=150, edgecolor='k')
plt.title('Semi-supervizovaný vstup')
plt.xlabel('Vlastnost 1')
plt.ylabel('Vlastnost 2')
plt.grid(True, alpha=0.3)

# Predikce modelu
plt.subplot(133)
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', 
           edgecolor='k', s=80, alpha=0.7)
plt.title(f'Predikce modelu\nPřesnost na neoznačených datech: {accuracy_unlabeled:.2f}')
plt.xlabel('Vlastnost 1')
plt.ylabel('Vlastnost 2')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Vytisknout klasifikační report
print("Klasifikační report pro neoznačená data:")
print(classification_report(y_true[unlabeled_idx], y_pred[unlabeled_idx]))

# Kontrola, jak se liší predikce od skutečných hodnot
diff_count = np.sum(y_pred != y_true)
print(f"Počet rozdílů mezi predikcemi a skutečnými značkami: {diff_count} z {len(y_true)}")

## 9. Supervizované učení

### Popis
Supervizované učení je paradigma strojového učení, kde se modely učí z označených trénovacích dat, aby mohly dělat predikce na nových, neviděných datech. Algoritmus se učí mapovací funkci mezi vstupními proměnnými a výstupními proměnnými na základě příkladů vstupů a výstupů. Proces učení pokračuje, dokud model nedosáhne přijatelné úrovně výkonu na trénovacích datech.

**Aplikace:** Detekce spamu v e-mailech, hodnocení úvěrového rizika a schvalování půjček, lékařská diagnostika z příznaků nebo snímků pacientů, analýza sentimentu textu, systémy doporučování produktů, prognóza prodejů a poptávky, rozpoznávání rukopisu a obrazů, rozpoznávání řeči

### Příklad: Víceúrovňová klasifikace kosatců

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import plot_tree

# Nastavení seedu pro reprodukovatelnost
np.random.seed(42)

# Načtení datasetu Iris (klasický dataset pro supervizované učení)
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Vytvoření DataFrame pro snadnější manipulaci a vizualizaci
df = pd.DataFrame(data=X, columns=feature_names)
df['species'] = [target_names[i] for i in y]

print("Přehled datasetu:")
print(f"- Počet vzorků: {X.shape[0]}")
print(f"- Počet vlastností: {X.shape[1]}")
print(f"- Třídy: {', '.join(target_names)}")
print("\nPrvních 5 řádků:")
print(df.head())

# Průzkumná analýza dat
plt.figure(figsize=(12, 10))

# Párový graf pro zobrazení vztahů mezi vlastnostmi
plt.subplot(221)
sns.scatterplot(x='sepal length (cm)', y='sepal width (cm)', 
                hue='species', data=df, palette='viridis')
plt.title('Rozměry kalichu podle druhů')

plt.subplot(222)
sns.scatterplot(x='petal length (cm)', y='petal width (cm)', 
                hue='species', data=df, palette='viridis')
plt.title('Rozměry okvětního lístku podle druhů')

# Distribuce vlastností
plt.subplot(223)
for species in target_names:
    sns.kdeplot(df[df['species'] == species]['petal length (cm)'], 
                label=species)
plt.title('Distribuce délky okvětního lístku podle druhů')
plt.legend()

plt.subplot(224)
sns.heatmap(df.drop('species', axis=1).corr(), annot=True, cmap='coolwarm')
plt.title('Korelační matice vlastností')

plt.tight_layout()
plt.show()

# Rozdělení dat na trénovací a testovací sady
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Škálování vlastností
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Trénování klasifikátoru rozhodovacího stromu (klasický algoritmus supervizovaného učení)
dt_model = DecisionTreeClassifier(random_state=42, max_depth=3)
dt_model.fit(X_train_scaled, y_train)

# Vytvoření predikcí
y_pred = dt_model.predict(X_test_scaled)

# Vyhodnocení modelu
accuracy = accuracy_score(y_test, y_pred)
print(f"Přesnost modelu: {accuracy:.4f}")
print("\nKlasifikační report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Vizualizace matice záměn
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Matice záměn')
plt.xlabel('Predikováno')
plt.ylabel('Skutečně')
plt.show()

# Vizualizace rozhodovacího stromu
plt.figure(figsize=(15, 10))
plot_tree(dt_model, feature_names=feature_names, class_names=target_names, filled=True, rounded=True, fontsize=10)
plt.title('Vizualizace rozhodovacího stromu')
plt.show()

# Křížová validace pro robustnější hodnocení výkonu modelu
cv_scores = cross_val_score(dt_model, X, y, cv=5)
print(f"\nPřesnost křížové validace: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Porovnání s jiným supervizovaným algoritmem (Random Forest)
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"\nPřesnost Random Forest: {rf_accuracy:.4f}")

# Důležitost vlastností
plt.figure(figsize=(10, 6))
importances = rf_model.feature_importances_
indices = np.argsort(importances)

plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Důležitost vlastnosti')
plt.title('Důležitost vlastností podle Random Forest')
plt.show()

# Analýza, kde model dělá chyby
incorrect_indices = np.where(y_test != y_pred)[0]
if len(incorrect_indices) > 0:
    print("\nPříklady, kde model udělal chyby:")
    X_test_df = pd.DataFrame(X_test, columns=feature_names)
    X_test_df['actual'] = [target_names[i] for i in y_test]
    X_test_df['predicted'] = [target_names[i] for i in y_pred]
    print(X_test_df.iloc[incorrect_indices])
else:
    print("\nModel správně klasifikoval všechny testovací vzorky!")

## Závěrečné rozhodnutí: Které paradigma je nejvíce používané v reálném ML

Po prozkoumání všech osmi paradigmat strojového učení je nejvíce používaným přístupem v reálných aplikacích **Supervizované učení**, zejména **Klasifikace** a **Regrese**.

### Důvody, proč supervizované učení dominuje v reálných ML aplikacích:

1. **Jasná formulace problému**: Problémy supervizovaného učení mají dobře definované vstupy a očekávané výstupy, což je činí snazšími pro formulaci a hodnocení.

2. **Interpretovatelnost**: Klasifikační a regresní modely často poskytují interpretovatelné výsledky ve srovnání s nesupervizovanými metodami, což je klíčové v mnoha obchodních, lékařských a právních aplikacích.

3. **Zavedené metodologie**: Existují robustní, dobře pochopené metodologie pro trénování, validaci a nasazení supervizovaných modelů.

4. **Obchodní hodnota**: Je snazší prokázat přímou obchodní hodnotu při předpovídání konkrétních cílů (prodeje, odchod zákazníků, detekce podvodů atd.).

5. **Různorodé algoritmy**: Existuje bohatý ekosystém algoritmů (od lineárních modelů po komplexní neuronové sítě), které lze aplikovat na základě velikosti dat, složitosti a výkonnostních požadavků.

6. **Podpora frameworků**: Hlavní ML frameworky poskytují rozsáhlou podporu pro úlohy supervizovaného učení.

To však neznamená, že ostatní paradigmata nejsou důležitá - každé má své místo v ML ekosystému:

- **Nesupervizované učení** (zejména clustering) je nezbytné pro průzkumnou analýzu dat a objevování skrytých vzorů.
- **Ensemble learning** se stal standardní praxí pro zvýšení výkonu v soutěžích a produkčních systémech.
- **Analýza časových řad** dominuje v aplikacích pro prognózování.
- **Hluboké učení** (podmnožina supervizovaného učení) způsobilo revoluci v oblastech jako počítačové vidění a zpracování přirozeného jazyka.

Volba paradigmatu nakonec závisí na:
- Povaze dostupných dat
- Konkrétním řešeném problému
- Výpočetních zdrojích
- Požadavcích na interpretovatelnost
- Potřebě predikce v reálném čase

S pokrokem v oboru pozorujeme rostoucí integraci více paradigmat v sofistikovaných ML systémech, přičemž semi-supervizované učení, přenosové učení a posilované učení získávají větší uplatnění v reálných aplikacích.